# Walkthrough: curating the offline Dolly sample

This notebook runs the **llm-dataset-curation** pipeline end-to-end on the
checked-in sample (`configs/sample.yaml`) — no network required.

**What you will see**

1. A short map of the stages
2. A Python-API run that writes curated / removed / report artifacts
3. Funnel and stats exploration
4. Concrete before/after examples (PII redaction, near-dupes, quality drops)

**Setup** (from the repo root):

```bash
uv sync --extra notebook   # optional: jupyter + ipykernel for interactive runs
uv run jupyter notebook notebooks/walkthrough.ipynb
```

Or execute cells in any IDE that can use the project `.venv`.


## Pipeline at a glance

| Stage | Role |
|---|---|
| `normalize` | Unicode NFKC, strip control chars, collapse whitespace |
| `pii` | Redact email / phone / IPv4 / card / SSN *before* anything else sees raw PII |
| `exact_dedupe` | SHA-256 identity over configured fields |
| `near_dedupe` | MinHash LSH + verified Jaccard clusters |
| `quality` | Gopher/C4-style heuristics + language ID |
| `toxicity` | Lexicon (optional classifier hook) |

Identity for instruction data is **`instruction + response`**, so shared context
passages alone are not treated as duplicates.


In [ ]:
import json
import os
from pathlib import Path

import polars as pl

from curate.config import load_config
from curate.pipeline import run_pipeline
from curate.report import write_report

# Notebook may be started from repo root or from notebooks/
ROOT = Path.cwd()
if not (ROOT / "configs" / "sample.yaml").exists():
    ROOT = ROOT.parent
cfg_path = ROOT / "configs" / "sample.yaml"
assert cfg_path.exists(), f"configs/sample.yaml not found from {Path.cwd()}"
os.chdir(ROOT)
print("repo root:", ROOT.resolve())

## Run the sample config

`load_config` + `run_pipeline` + `write_report` is the same path the CLI uses
(`curate run --config configs/sample.yaml`). Outputs land under
`output/dolly-sample/` (overridden here so re-running the notebook is idempotent).


In [ ]:
OUT = ROOT / "output" / "notebook-walkthrough"

cfg = load_config(
    ROOT / "configs" / "sample.yaml",
    {"output": {"dir": str(OUT)}},
)
cfg.base_dir = ROOT

run = run_pipeline(cfg, write=True)
report_paths = write_report(run)

print(
    f"{cfg.name}: {run.input_df.height:,} → {run.curated.height:,} records "
    f"(retention {run.curated.height / max(run.input_df.height, 1):.1%})"
)
print("out_dir:", run.out_dir)
for kind, path in report_paths.items():
    print(f"  {kind}: {path}")

## Funnel and stage stats

Every removed row keeps `_stage` and a human-readable `_reason`. `stats.json`
mirrors the funnel the report charts.


In [ ]:
stats = json.loads((run.out_dir / "stats.json").read_text())

funnel_df = pl.DataFrame(stats["funnel"])
print(funnel_df)

print("\nremoved by stage:")
print(run.removed.group_by("_stage").len().sort("len", descending=True))

print("\nper-stage timing (seconds):")
for name, s in stats["stages"].items():
    print(f"  {name:<13} {s.get('seconds', '?'):>6}")

## Load artifacts from disk

After a run you can reopen the parquet / JSON artifacts without re-curating.


In [ ]:
curated = pl.read_parquet(run.out_dir / "curated.parquet")
removed = pl.read_parquet(run.out_dir / "removed.parquet")
clusters_path = run.out_dir / "dup_clusters.parquet"
clusters = pl.read_parquet(clusters_path) if clusters_path.exists() else None

print("curated columns:", curated.columns)
print("removed columns:", removed.columns)
print("curated rows:", curated.height, "| removed rows:", removed.height)
if clusters is not None:
    print(
        "near-dup cluster edges:",
        clusters.height,
        "| unique clusters:",
        clusters["cluster_id"].n_unique(),
    )

# Spot-check report markdown exists and mentions the funnel section
md = (run.out_dir / "report.md").read_text(encoding="utf-8")
assert "## Funnel" in md
print("report.md bytes:", len(md))

## Before / after: PII redaction

PII is redacted in place (records stay in the curated set unless
`action: drop`). Synthetic noise rows use fake `example.com` / `555` values.


In [ ]:
# Pick a labelled synthetic PII row that survived (redacted, not dropped)
pii_id = "synthetic-022"
before = run.input_df.filter(pl.col("id") == pii_id).row(0, named=True)
after = curated.filter(pl.col("id") == pii_id).row(0, named=True)

print("id:", pii_id)
print("pii_hits / types:", after.get("pii_hits"), after.get("pii_types"))
print("\nBEFORE response:\n ", before["response"])
print("\nAFTER  response:\n ", after["response"])
assert "@example.com" not in after["response"]
assert "<EMAIL>" in after["response"] and "<PHONE>" in after["response"]

## Before / after: near-duplicate removal

Near-dedupe keeps the earliest record in a MinHash cluster and drops the rest
with a reason that names the representative and estimated Jaccard.


In [ ]:
near = removed.filter(pl.col("_stage") == "near_dedupe")
assert near.height > 0, "expected near-dup removals on the sample"

example = near.head(1).row(0, named=True)
reason = example["_reason"]  # e.g. "near duplicate of dolly-01279 (J≈0.95)"
rep_id = reason.split("of ", 1)[1].split(" ", 1)[0]
rep = run.input_df.filter(pl.col("id") == rep_id).row(0, named=True)

print("removed:", example["id"])
print("reason: ", reason)
print("\nrepresentative instruction:\n ", rep["instruction"][:200])
print("\nduplicate instruction:\n ", (example.get("instruction") or "")[:200])

## Quality and toxicity drops

A few removed rows with their stage reasons — useful when tuning thresholds.


In [ ]:
def show_removed(stage: str, n: int = 3) -> None:
    rows = removed.filter(pl.col("_stage") == stage).head(n)
    print(
        f"\n=== {stage} ({rows.height} shown / "
        f"{removed.filter(pl.col('_stage') == stage).height} total) ==="
    )
    for r in rows.to_dicts():
        instr = (r.get("instruction") or r.get("text") or "")[:100].replace("\n", " ")
        print(f"- {r['id']}: {r['_reason']}")
        print(f"  instruction: {instr}")


for stage in ("exact_dedupe", "quality", "toxicity"):
    if stage in removed["_stage"].to_list():
        show_removed(stage)

## Eval metrics on labelled synthetic noise

When the input carries `noise_type` labels (as the sample does), `stats["eval"]`
reports per-stage precision / recall against those labels.


In [ ]:
eval_block = stats.get("eval", {})
per_stage = eval_block.get("per_stage", {})
rows = []
for stage, m in per_stage.items():
    rows.append(
        {
            "stage": stage,
            "precision": m.get("precision"),
            "recall": m.get("recall"),
            "support": m.get("support"),
            "tp": m.get("tp"),
            "fp": m.get("fp"),
            "fn": m.get("fn"),
        }
    )
print(pl.DataFrame(rows).sort("stage"))

## Next steps

- Open `report.html` in a browser for charts and sampled removals.
- Point a copy of `configs/sample.yaml` at your own JSONL / Parquet / CSV
  (`input.paths`, `id_field`, `text_fields`).
- Override thresholds from the CLI:
  `uv run curate run -c configs/sample.yaml --set stages.near_dedupe.threshold=0.7`
- Full Dolly-15k: `make full` (downloads ~13 MB once).
